In [ ]:
import os
from pathlib import Path

# Define the path to your single PDF document
YOUR_DOCUMENT_PATH = "path/to/your/document.pdf"

# Experiment: Finding the Optimal Chunk Size

This notebook outlines an experiment to determine the most effective chunk size for a Retrieval-Augmented Generation (RAG) system based on a single PDF document. The quality of the retrieval is a critical factor in the performance of the RAG system, and the chunk size is a key parameter that influences it.

## Objective
The primary goal is to empirically find the optimal `chunk_size` that provides the most relevant context for answering a variety of questions about the document.

## Methodology

The experiment will follow these steps:

1.  **Document Loading**: A single PDF document is loaded and its text content is extracted.

2.  **Corpus Analysis**: The script performs a basic analysis of the document, calculating the total number of characters, tokens, and sentences. This helps in understanding the scale of the document and provides a basis for selecting a range of chunk sizes to test.

3.  **Question Generation**: A set of domain-specific questions is generated using a language model (Mistral-7B). These questions are designed to be representative of what a user might ask and will serve as the basis for evaluation.

4.  **Chunking and Indexing**: The document is split into chunks using different `chunk_size` values (e.g., 128, 256, 512, 1024). For each chunk size, a separate vector store index is created.

5.  **Retrieval and Evaluation**: For each indexed chunking strategy:
    *   The generated questions are used to query the retriever.
    *   The relevance and quality of the retrieved chunks are evaluated. This can be done using metrics like `Hit Rate` (did the retriever find the correct chunk?) or by using a more advanced evaluation framework like Ragas, which assesses `Context Precision` and `Context Recall`.

6.  **Analysis**: The evaluation scores for each chunk size are compared. The `chunk_size` that yields the highest average score across all questions will be considered the most optimal for this particular document and use case.

By the end of this experiment, we will have data-driven evidence to select the best chunking strategy for our RAG pipeline.

In [ ]:
import PyPDF2
from pathlib import Path
from llama_index.core import SimpleDirectoryReader

def load_pdf(file_path: str) -> str:
    """Load PDF file"""
    text = ""
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        if reader.is_encrypted:
            try:
                reader.decrypt('')
            except Exception as e:
                print(f"Could not decrypt PDF: {e}")
                # Fallback for encrypted or complex PDFs
                docs = SimpleDirectoryReader(input_files=[file_path]).load_data()
                return "\n\n".join([doc.text for doc in docs])

        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

# Load your document
corpus_text = load_pdf(YOUR_DOCUMENT_PATH)

print(f"Loaded corpus with {len(corpus_text)} characters")

In [ ]:
def analyze_corpus(text: str):
    """Analyze corpus to determine appropriate parameters"""
    
    num_chars = len(text)
    num_tokens = count_tokens(text)
    num_sentences = len(nltk.sent_tokenize(text))
    
    print(f"Corpus Analysis:")
    print(f"  Characters: {num_chars:,}")
    print(f"  Tokens: {num_tokens:,}")
    print(f"  Sentences: {num_sentences:,}")
    print(f"\nRecommendations:")
    
    # Estimate number of chunks for different sizes
    for chunk_size in [128, 256, 512, 1024, 2048]:
        est_chunks = num_tokens // chunk_size
        print(f"  Chunk size {chunk_size}: ~{est_chunks} chunks")
    
    # Recommend number of queries
    recommended_queries = min(max(est_chunks * 2, 50), 500)
    print(f"\nRecommended number of queries: {recommended_queries}")
    
    return {
        'num_tokens': num_tokens,
        'num_sentences': num_sentences,
        'recommended_queries': recommended_queries
    }

corpus_stats = analyze_corpus(corpus_text)

In [ ]:
def validate_document_size(text: str, min_tokens: int = 5000):
    """Ensure document is large enough for meaningful chunking experiments"""
    
    num_tokens = count_tokens(text)
    
    if num_tokens < min_tokens:
        print(f"⚠️  WARNING: Document only has {num_tokens} tokens")
        print(f"   Recommended minimum: {min_tokens} tokens")
        print(f"   Consider:")
        print(f"   - Adding more documents")
        print(f"   - Testing smaller chunk sizes (64, 128, 256)")
        print(f"   - Combining multiple related documents")
        return False
    
    return True

# Validate before proceeding
if not validate_document_size(corpus_text):
    print("\nProceeding anyway, but results may not be meaningful...")

In [ ]:
def sample_large_corpus(text: str, max_tokens: int = 100000):
    """
    For very large corpora, sample a representative subset
    (Similar to paper using "first 60 pages")
    """
    
    num_tokens = count_tokens(text)
    
    if num_tokens > max_tokens:
        print(f"📊 Corpus has {num_tokens:,} tokens")
        print(f"   Sampling first {max_tokens:,} tokens for efficiency...")
        
        # Sample by token count (approximate)
        encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
        tokens = encoding.encode(text)
        sampled_tokens = tokens[:max_tokens]
        sampled_text = encoding.decode(sampled_tokens)
        
        # Try to end at sentence boundary
        sentences = nltk.sent_tokenize(sampled_text)
        sampled_text = " ".join(sentences[:-1])  # Remove partial last sentence
        
        print(f"   Sampled to {count_tokens(sampled_text):,} tokens")
        return sampled_text
    
    return text

# Apply sampling if needed
corpus_text = sample_large_corpus(corpus_text, max_tokens=100000)

In [ ]:
def generate_domain_specific_queries(corpus_text, domain_context="", num_queries=170):
    """
    Generate queries specific to your document domain
    
    Args:
        corpus_text: Your document text
        domain_context: Description of document type (e.g., "medical research paper", 
                       "financial report", "technical documentation")
        num_queries: Number of queries to generate
    """
    
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch
    
    model_name = "mistralai/Mistral-7B-Instruct-v0.2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # Domain-specific prompt
    if domain_context:
        domain_instruction = f"This is a {domain_context}."
    else:
        domain_instruction = "This is a professional document."
    
    prompt = f"""<s>[INST] {domain_instruction}

Based on the following document excerpt, generate {num_queries} realistic questions that users might ask about this document.

Document excerpt:
{corpus_text[:3000]}

Generate diverse questions including:
- Specific facts and details
- Definitions and explanations
- Numerical data and statistics
- Comparisons and relationships
- Summaries and overviews
- "How" and "Why" questions
- "What is the difference between" questions

Format each question on a new line, numbered.

Questions: [/INST]

1."""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=2500,
        temperature=0.8,
        do_sample=True,
        top_p=0.95
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract questions
    questions = []
    lines = response.split('\n')
    
    for line in lines:
        line = line.strip()
        # Match numbered questions
        if line and any(line.startswith(f"{i}.") for i in range(1, 300)):
            # Remove numbering
            question = line.split('.', 1)[1].strip() if '.' in line else line
            if question and '?' in question:
                questions.append(question)
    
    return questions[:num_queries]

# Example usage for different domains
# For financial documents:
queries = generate_domain_specific_queries(
    corpus_text, 
    domain_context="financial annual report",
    num_queries=170
)

# For technical documentation:
# queries = generate_domain_specific_queries(
#     corpus_text,
#     domain_context="software technical documentation",
#     num_queries=170
# )

# For medical/research:
# queries = generate_domain_specific_queries(
#     corpus_text,
#     domain_context="medical research paper",
#     num_queries=170
# )